# Module 02 — Custom Tools with `@tool`

An agent without tools is just a chatbot. Tools are what make agents *act* — they let the model call Python functions to fetch data, run calculations, call APIs, and more.

### How tools work in Strands

1. You decorate a Python function with `@tool`
2. Strands reads the **docstring** and **type hints** to generate a JSON schema
3. The model decides when to call the tool based on the user's request
4. Strands executes the function and feeds the result back to the model
5. The model uses the result to form its final response

The docstring is critical — it tells the model *what the tool does* and *when to use it*.

In [3]:
import requests
from strands import Agent, tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city using the Open-Meteo API.
    
    Args:
        city: The name of the city to get weather for
    
    Returns:
        A string describing the current weather including temperature and conditions
    """
    try:
        # Step 1: Geocode the city name to lat/lon
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_resp = requests.get(geo_url, params={"name": city, "count": 1}, timeout=10)
        geo_resp.raise_for_status()
        geo_data = geo_resp.json()
        
        if not geo_data.get("results"):
            return f"Could not find location: {city}"
        
        result = geo_data["results"][0]
        lat = result["latitude"]
        lon = result["longitude"]
        name = result["name"]
        country = result.get("country", "")
        
        # Step 2: Fetch current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "temperature_unit": "celsius",
            "wind_speed_unit": "kmh",
        }
        weather_resp = requests.get(weather_url, params=params, timeout=10)
        weather_resp.raise_for_status()
        weather_data = weather_resp.json()
        
        current = weather_data["current"]
        temp = current["temperature_2m"]
        humidity = current["relative_humidity_2m"]
        wind = current["wind_speed_10m"]
        
        # WMO weather code to description
        wmo_codes = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 48: "Icy fog", 51: "Light drizzle", 53: "Drizzle",
            61: "Light rain", 63: "Rain", 65: "Heavy rain",
            71: "Light snow", 73: "Snow", 75: "Heavy snow",
            80: "Light showers", 81: "Showers", 82: "Heavy showers",
            95: "Thunderstorm",
        }
        condition = wmo_codes.get(current["weather_code"], f"Code {current['weather_code']}")
        
        return (
            f"{name}, {country}: {condition}, "
            f"{temp}°C, Humidity: {humidity}%, Wind: {wind} km/h"
        )
    
    except Exception as e:
        return f"Error fetching weather for {city}: {e}"


agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[get_weather]
)

response = agent("What's the current weather in Mysuru, Puttur, and Madikeri?")
print(response)



Tool #1: get_weather

Tool #2: get_weather

Tool #3: get_weather
Here's the current weather for those three cities in India:

**Mysuru:**
- Condition: Overcast
- Temperature: 24.8°C
- Humidity: 78%
- Wind: 9.1 km/h

**Puttur:**
- Condition: Overcast
- Temperature: 26.4°C
- Humidity: 80%
- Wind: 6.8 km/h

**Madikeri:**
- Condition: Overcast
- Temperature: 21.0°C
- Humidity: 97%
- Wind: 2.1 km/h

All three locations are experiencing overcast conditions. Puttur is the warmest at 26.4°C, while Madikeri is the coolest at 21.0°C with notably higher humidity. Mysuru has the strongest winds among the three.Here's the current weather for those three cities in India:

**Mysuru:**
- Condition: Overcast
- Temperature: 24.8°C
- Humidity: 78%
- Wind: 9.1 km/h

**Puttur:**
- Condition: Overcast
- Temperature: 26.4°C
- Humidity: 80%
- Wind: 6.8 km/h

**Madikeri:**
- Condition: Overcast
- Temperature: 21.0°C
- Humidity: 97%
- Wind: 2.1 km/h

All three locations are experiencing overcast conditions. Pu

The model automatically decided to call `get_weather` with `city="Mysuru, Bengaluru, and Madikeri"`, got the result, and used it in its response.

---

### Multiple tools

You can give an agent as many tools as it needs. The model picks the right one.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression safely.
    
    Args:
        expression: A math expression like '2 + 2' or '10 * 5 / 2'
    
    Returns:
        The result as a string
    """
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/(). ')
        if not all(c in allowed for c in expression):
            return "Error: only basic math operators allowed"
        result = eval(expression)  # safe because we validated input
        return str(result)
    except Exception as e:
        return f"Error: {e}"


@tool
def get_time() -> str:
    """Get the current date and time.
    
    Returns:
        Current date and time as a string
    """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[get_weather, calculate, get_time]
)

# The agent will use the right tool for each question
agent("What time is it?")
agent("What is 123 * 456?")
response = agent("What's the weather in Puttur, and what is 100 divided by 4?")
print(response)


Tool #1: get_time
The current date and time is **May 8, 2026 at 8:47:45 PM** (20:47:45).
Tool #2: calculate
123 * 456 = **56,088**
Tool #3: get_weather

Tool #4: calculate
Here's the information you requested:

**Weather in Mysuru, India:**
- Conditions: Overcast
- Temperature: 24.8°C
- Humidity: 78%
- Wind: 9.1 km/h

**100 divided by 4 = 25**Here's the information you requested:

**Weather in Mysuru, India:**
- Conditions: Overcast
- Temperature: 24.8°C
- Humidity: 78%
- Wind: 9.1 km/h

**100 divided by 4 = 25**



### Using built-in tools from strands-agents-tools

The `strands-agents-tools` package ships with ready-made tools for common tasks.

In [5]:
# strands-agents-tools includes: http_request, python_repl, file_read, shell, and more
from strands_tools import http_request

agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[http_request]
)

response = agent("Fetch https://httpbin.org/json and tell me what's in the response.")
print(response)


Tool #1: http_request


╭────────────────────────────────────── 🚀 HTTP Request Preview: GET /json ───────────────────────────────────────╮
│                                                                                                                 │
│   Method    GET                                                                                                 │
│   URL       https://httpbin.org/json                                                                            │
│   Headers   {}                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending request...

╭─────────────────────────────────────────── ✅ HTTP Response: 200 OK ────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│     Status         200 OK                                                                                       │
│     URL            https://httpbin.org/json                                                                     │
│     Content-Type   application/json                                                                             │
│     Size           429 bytes                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                          Response Headers                          
╭──────────────────────────────────┬───────────────────────────────╮
│ Header                           │ Value                         │
├──────────────────────────────────┼───────────────────────────────┤
│ Date                             │ Fri, 08 May 2026 15:18:44 GMT │
│ Content-Type                     │ application/json              │
│ Content-Length                   │ 429                           │
│ Connection                       │ keep-alive                    │
│ Server                           │ gunicorn/19.9.0               │
│ Access-Control-Allow-Origin      │ *                             │
│ Access-Control-Allow-Credentials │ true                          │
╰──────────────────────────────────┴───────────────────────────────╯

The response from `https://httpbin.org/json` contains a JSON object describing a sample slideshow. Here's what's in it:

**Structure:**
- **Author**: "Yours Truly"
- **Date**: "date of publication"
- **Title**: "Sample Slide Show"

**Slides**: The slideshow contains 2 slides:

1. **First Slide**:
   - Title: "Wake up to WonderWidgets!"
   - Type: "all"

2. **Second Slide**:
   - Title: "Overview"
   - Type: "all"
   - Items:
     - "Why <em>WonderWidgets</em> are great"
     - "Who <em>buys</em> WonderWidgets"

The response was returned with HTTP status code 200 (successful) and is properly formatted as JSON. It appears to be sample data used for testing JSON endpoints.The response from `https://httpbin.org/json` contains a JSON object describing a sample slideshow. Here's what's in it:

**Structure:**
- **Author**: "Yours Truly"
- **Date**: "date of publication"
- **Title**: "Sample Slide Show"

**Slides**: The slideshow contains 2 slides:

1. **First Slide**:
   - Title: "Wake up to 

---

### Key takeaways

- Decorate any Python function with `@tool` to make it available to the agent
- The **docstring** is the tool's description — write it clearly
- **Type hints** on parameters are required — they generate the JSON schema
- Pass tools as a list: `Agent(tools=[tool1, tool2])`
- The model decides when and which tool to call

Next up: **03_system_prompt.ipynb** — shaping agent behavior with system prompts.